# SEVERA: Road Accident Severity Early-Warning System
### End-to-End Machine Learning Pipeline & Explainability

This notebook demonstrates:
1. **Data Ingestion & Exploration** (50,000 accident records)
2. **Feature Engineering** (temporal extraction, rush hour, night, wet-high-speed flags)
3. **Handling Extreme Class Imbalance** (Cost-sensitive weighting + SMOTE)
4. **Benchmarking 4 Algorithms**: Logistic Regression, Random Forest, XGBoost, LightGBM
5. **Recall-Optimized Evaluation** for the **Fatal** minority class
6. **SHAP (SHapley Additive exPlanations)** global & local interpretability

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import shap

# Set display options
pd.set_option('display.max_columns', 20)
print("Environment initialized.")

## 1. Load Dataset & Inspect Distributions

In [ ]:
data_path = '../data/synthetic_accidents.csv'
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Check Class Imbalance

In [ ]:
severity_counts = df['severity'].value_counts(normalize=True) * 100
print("Severity distribution (%):")
print(severity_counts)

# Note: Fatal accidents comprise ~2% of accidents, creating an extreme imbalance challenge.

## 3. Run Training Pipeline & Model Comparison

In [ ]:
import sys
sys.path.append('..')
from train import train_and_evaluate

# Train all 4 models and save best artifact
train_and_evaluate()

## 4. Inspect Evaluation Metrics & Fatal Recall Optimization

> **Why optimize for Fatal Recall?**
> In emergency response triage, the cost of a False Negative (missing an actual Fatal accident) is measured in human lives. In contrast, a False Positive (over-triage) mobilizes advanced medical dispatch which can be stood down once on scene.

In [ ]:
with open('../metrics.json', 'r') as f:
    metrics = json.load(f)

print(f"Best model selected: {metrics['best_model']}")
for model_name, m in metrics['models'].items():
    print(f"\n--- {model_name} ---")
    print(f"Accuracy: {m['accuracy']} | Fatal Recall: {m['fatal_recall']} | Fatal Prec: {m['per_class']['Fatal']['precision']} | Macro F1: {m['macro_f1']}")

## 5. SHAP Feature Importance Analysis

In [ ]:
top_shaps = pd.Series(metrics['shap_importance'])
print("Top SHAP features driving severity:")
print(top_shaps.head(10))